In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import optuna
from pathlib import Path


In [ ]:
BASE_DIR = Path().resolve().parent
data_path = BASE_DIR / "data" / "mushrooms_clean.csv"
df = pd.read_csv(data_path)
df.head()

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

dividir entre entrenar modelo y hacer test 80/20%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

hacerlas categoricas onehotencoder

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), X.columns)
])

In [ ]:
rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_pipeline.fit(X_train, y_train)

gb_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(X_train, y_train)

In [ ]:
def check_overfitting(model):
    train = model.score(X_train, y_train)
    test = model.score(X_test, y_test)
    return train, test, train - test

rf_train, rf_test, rf_over = check_overfitting(rf_pipeline)
gb_train, gb_test, gb_over = check_overfitting(gb_pipeline)

print(f"RF: train={rf_train:.6f}, test={rf_test:.6f}, overfitting={rf_over:.6f}")
print(f"GB: train={gb_train:.6f}, test={gb_test:.6f}, overfitting={gb_over:.6f}")

In [ ]:
cv_rf = cross_val_score(rf_pipeline, X, y, cv=5, scoring="accuracy")

print(
    f"RF CV Accuracy: {cv_rf.mean():.6f} Â± {cv_rf.std():.6f}"
)

In [ ]:
logreg_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logreg_pipeline.fit(X_train, y_train)

y_pred_lr = logreg_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_lr))

lr_train = logreg_pipeline.score(X_train, y_train)
lr_test = logreg_pipeline.score(X_test, y_test)

print(f"Train: {lr_train:.6f}")
print(f"Test: {lr_test:.6f}")
print(f"Overfitting: {lr_train - lr_test:.6f}")

cv_lr = cross_val_score(
    logreg_pipeline,
    X,
    y,
    cv=5,
    scoring='f1'
)

print(f"CV F1 Mean: {cv_lr.mean():.6f}")

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Train Accuracy": [
        lr_train,
        rf_train,
        gb_train
    ],
    "Test Accuracy": [
        lr_test,
        rf_test,
        gb_test
    ]
})

results["Overfitting"] = (
    results["Train Accuracy"] - results["Test Accuracy"]
)

results